In [ ]:
import io
from PIL import Image

# Compress in memory
img = Image.open("Galaxy.jpg")
img.thumbnail((1024, 1024))  # Scale max dimensions
buf = io.BytesIO()
img.save(buf, format="JPEG", quality=70)
byte_array = buf.getvalue()

print(f"Compressed size: {len(byte_array)} bytes")
img.save("compressed_input.jpg", format="JPEG", quality=70)

In [ ]:
### MQTT Image Publisher ###

# Reads the image file in binary mode ("rb") and transmits the byte array over a topic.

# Practical Considerations

    # Broker Payload Limits: Default Mosquitto installations typically restrict message 
    # size via message_size_limit (often default ~1 MB or up to ~256 MB if configured). If you encounter dropped packets on high-resolution images, compress the image first or chunk the payload.

    # Avoid Base64: Sending raw binary via msg.payload avoids the ~33% size 
    # inflation and CPU overhead of Base64 encoding.
    
# Helps to compress the original image for fast and reliable transmission

import io
import threading
from PIL import Image
import paho.mqtt.client as mqtt

BROKER = "test.mosquitto.org"
PORT = 1883
TOPIC = TOPIC = "my_custom_image_feed_2026/camera/image"
INPUT_IMAGE_PATH = "Galaxy.jpg"

published_event = threading.Event()

def on_connect(client, userdata, flags, reason_code, properties=None):
    if reason_code == 0:
        print("Connected to broker.")
    else:
        print(f"Connection failed: {reason_code}")

def on_publish(client, userdata, mid, reason_code=None, properties=None):
    print("Broker acknowledged receipt.")
    published_event.set()

# 1. Open, convert color format, resize, and compress
with Image.open(INPUT_IMAGE_PATH) as img:
    img = img.convert("RGB")
    img.thumbnail((1024, 1024))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=70)
    byte_array = buf.getvalue()

print(f"Compressed byte size: {len(byte_array)} bytes")

# 2. Setup MQTT client
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="in_memory_publisher")
client.on_connect = on_connect
client.on_publish = on_publish

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()

# 3. Publish and wait for broker confirmation
client.publish(TOPIC, payload=byte_array, qos=1, retain=True)

# Wait up to 10 seconds for the broker acknowledgment
success = published_event.wait(timeout=10)
if success:
    print(f"Successfully transmitted {len(byte_array)} bytes.")
else:
    print("Publish timed out before broker acknowledged.")

client.loop_stop()
client.disconnect()

In [ ]:
### MQTT Image Subscriber ###

# Subscribes to the topic and saves msg.payload directly to disk using binary write mode ("wb").

import paho.mqtt.client as mqtt

BROKER = "test.mosquitto.org"
PORT = 1883
TOPIC = TOPIC = "my_custom_image_feed_2026/camera/image"
OUTPUT_PATH = "received.jpg"

def on_connect(client, userdata, flags, reason_code, properties=None):
    print("Connected. Subscribing to topic...")
    client.subscribe(TOPIC, qos=1)

def on_message(client, userdata, msg):
    print(f"Received {len(msg.payload)} bytes. Writing to {OUTPUT_PATH}...")
    with open(OUTPUT_PATH, "wb") as file:
        file.write(msg.payload)
    print("Image saved successfully.")

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="image_subscriber")
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, keepalive=60)

# Block and listen for incoming messages
client.loop_forever()

In [ ]:
# Publisher of data

import paho.mqtt.client as mqtt 
from random import randrange, uniform
import time

mqttBroker = "test.mosquitto.org"
client = mqtt.Client()


client.connect(mqttBroker,port=1883) 

while True:
    randNumber = uniform(20.0, 21.0)
    client.publish("TEMPERATURE", randNumber)
    print("Just published " + str(randNumber) + " to topic TEMPERATURE")
    time.sleep(1)


# Initialize the MQTT client with the specified callback API version if using callback
# client = mqtt.Client(
#     mqtt.CallbackAPIVersion.VERSION2,  # Specify the API version
#     client_id="Temperature_Inside",    # Your client ID
#     protocol=mqtt.MQTTv311             # Optional: Specify MQTT protocol version
# )

In [ ]:
import paho.mqtt.client as mqtt #import the client1
import time

# Broker address
broker_address="test.mosquitto.org"     

# Connect to client
client = mqtt.Client() #create new instance

msg = [] # Container to store message data

# The callback for when the client receives a CONNACK response from the server
def on_connect(client, userdata, flags, rc):
    print(f"Connected with result code {rc}")
    # Subscribe to the topic 'TEMPERATURE'
    client.subscribe("TEMPERATURE")
    print("Subscribing to topic","TEMPERATURE")

# The callback for when a PUBLISH message is received from the server
def on_message(client, userdata, message):
    msg.append(message.payload.decode("utf-8"))
    print("message received " ,str(message.payload.decode("utf-8")))
    print("message topic=",message.topic)
    print("message qos=",message.qos)
    print("message retain flag=",message.retain)

def on_log(client, userdata, level, buf):
    print("log: ",buf)
    

# Attach the callback functions
client.on_connect = on_connect
client.on_message=on_message 

print("connecting to broker")
client.connect(broker_address,port=1883) #connect to broker

client.loop_forever() #start the loop

